# Topic 1: Delta Lake — ACID Transactions & Time Travel

> **Phase 5 → Week 9 → Topic 1**
>
> Prerequisites: Week 8 (Reading/Writing, Incremental Patterns)

---

## What is Delta Lake?

Delta Lake is an open-source storage layer on top of Parquet that adds:

```
Plain Parquet:                    Delta Lake:
  /data/table/                      /data/table/
    part-000.parquet                  part-000.parquet
    part-001.parquet                  part-001.parquet
    part-002.parquet          +       _delta_log/
                                        00000000000000000000.json  ← transaction log
                                        00000000000000000001.json
                                        00000000000000000002.json
                                        00000000000000000010.checkpoint.parquet

The _delta_log is what makes Delta Lake special:
  ✓ ACID transactions    (atomic writes — all or nothing)
  ✓ Time travel          (query any previous version)
  ✓ Audit log            (who wrote what, when)
  ✓ Schema enforcement   (reject writes that don't match the schema)
  ✓ Schema evolution     (add columns safely with ALTER TABLE)
  ✓ MERGE (upsert)       (UPDATE + INSERT in one atomic operation)
  ✓ OPTIMIZE + Z-ORDER   (compact files, co-locate related data)
```

---

## Interview Cheat Sheet

**Q: What is the Delta Lake transaction log?**
> The `_delta_log` directory contains a JSON file for every transaction. Each log entry records which files were added/removed. Spark constructs the current state of the table by replaying the log from the latest checkpoint. This is what enables time travel (replay only N entries), ACID (a transaction is one atomic log entry), and concurrent writes (optimistic concurrency control via log conflicts).

**Q: What is Delta Lake time travel?**
> Time travel lets you query a Delta table at a previous version or timestamp. `spark.read.format('delta').option('versionAsOf', 3).load('/path')` reads the table as it was at version 3. `option('timestampAsOf', '2024-01-15')` reads at a specific timestamp. Useful for: auditing, debugging wrong writes, reproducing ML training data, and rolling back bad updates.

**Q: What is VACUUM in Delta Lake?**
> `VACUUM` deletes old Parquet files that are no longer referenced by any Delta log version within the retention period (default 7 days). Without VACUUM, old files accumulate indefinitely. After VACUUM, you can no longer time-travel to versions older than the retention threshold. Always run VACUUM regularly in production to control storage costs.

**Q: What is OPTIMIZE and Z-ORDER in Delta Lake?**
> `OPTIMIZE` compacts many small Parquet files into fewer large files (target ~1GB each), improving read performance. `ZORDER BY (col)` co-locates related data in the same files by multi-dimensional clustering, so queries filtering on that column read fewer files. OPTIMIZE + ZORDER together = fewer files, each containing co-located data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os, shutil
from datetime import datetime

spark = SparkSession.builder \
    .appName("Week9 - Delta Lake") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Delta Lake requires JARs not in this container.
# We simulate versioning with per-version Parquet paths.
# In production: replace .parquet(...) with .format("delta").save(...)

BASE_PATH = "/tmp/sim_delta"
_version_log = []   # [{version, operation, timestamp, rows}]
_version = [-1]     # mutable current version counter

def _vpath(n=None):
    v = n if n is not None else _version[0]
    return f"{BASE_PATH}/v{v}"

def sim_write(df, mode="overwrite", label=None):
    """Save a new versioned snapshot and advance the version counter."""
    v = _version[0] + 1
    path = f"{BASE_PATH}/v{v}"
    df.write.mode("overwrite").parquet(path)
    _version[0] = v
    op = label or mode.upper()
    _version_log.append({
        "version": v,
        "operation": op,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "rows": df.count()
    })
    return v

def sim_read(version=None):
    """Read the latest (or a specific) version snapshot."""
    v = version if version is not None else _version[0]
    return spark.read.parquet(f"{BASE_PATH}/v{v}")

def sim_history():
    print("{:<8} {:<22} {:<12} {}".format("version", "timestamp", "operation", "rows"))
    print("-" * 50)
    for e in _version_log:
        print("{:<8} {:<22} {:<12} {}".format(
            e["version"], e["timestamp"], e["operation"], e["rows"]))

if os.path.exists(BASE_PATH):
    shutil.rmtree(BASE_PATH)
os.makedirs(BASE_PATH)

print("Simulation ready. BASE_PATH:", BASE_PATH)


## Part 1: Creating a Delta Table

In [ ]:
# Version 0: initial data
# Delta equivalent: orders_v0.write.format("delta").mode("overwrite").save(BASE_PATH)
orders_v0 = spark.createDataFrame([
    ("O001", "C001", 250.0, "pending"),
    ("O002", "C002",  89.5, "pending"),
    ("O003", "C001", 500.0, "pending"),
], ["order_id", "customer_id", "amount", "status"])

sim_write(orders_v0, label="WRITE")
print("Version 0 written — initial 3 orders")
sim_read().show()


In [ ]:
# Version 1: append 2 new orders
# Delta: orders_v1.write.format("delta").mode("append").save(BASE_PATH)
orders_v1 = spark.createDataFrame([
    ("O004", "C003",  45.0, "pending"),
    ("O005", "C002", 300.0, "pending"),
], ["order_id", "customer_id", "amount", "status"])

v1_full = sim_read().union(orders_v1)
sim_write(v1_full, label="APPEND")
print("Version 1 — appended 2 orders")

# Version 2: overwrite with updated statuses
# Delta: orders_v2.write.format("delta").mode("overwrite").save(BASE_PATH)
orders_v2 = spark.createDataFrame([
    ("O001", "C001", 250.0, "delivered"),
    ("O002", "C002",  89.5, "delivered"),
    ("O003", "C001", 500.0, "pending"),
    ("O004", "C003",  45.0, "cancelled"),
    ("O005", "C002", 300.0, "pending"),
], ["order_id", "customer_id", "amount", "status"])

sim_write(orders_v2, label="OVERWRITE")
print("Version 2 — overwrite with updated statuses")

print("\nCurrent state (version 2):")
sim_read().show()


## Part 2: Time Travel

In [ ]:
# Delta: dt = DeltaTable.forPath(spark, BASE_PATH)
#         dt.history().select("version","timestamp","operation").show()
print("Simulated Delta transaction history:")
sim_history()

# Time travel by version
# Delta: spark.read.format("delta").option("versionAsOf", 0).load(BASE_PATH).show()
print("\nVersion 0 (3 orders, all pending):")
sim_read(0).show()

print("Version 1 (5 orders, all pending):")
sim_read(1).show()

print("Version 2 (5 orders, some delivered/cancelled):")
sim_read(2).show()


## Part 3: MERGE (Upsert)

In [ ]:
# MERGE (upsert): update matched rows, insert new rows
# Delta:
#   dt.alias("target").merge(incoming.alias("source"), "source.order_id = target.order_id")
#     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

incoming = spark.createDataFrame([
    ("O001", "C001", 250.0, "delivered"),
    ("O003", "C001", 500.0, "delivered"),
    ("O006", "C004", 175.0, "pending"),
    ("O007", "C002",  60.0, "pending"),
], ["order_id", "customer_id", "amount", "status"])

current = sim_read()

unchanged   = current.join(incoming.select("order_id"), on="order_id", how="left_anti")
updated     = incoming.join(current.select("order_id"), on="order_id", how="inner")
new_rows    = incoming.join(current.select("order_id"), on="order_id", how="left_anti")

merged = unchanged.union(updated).union(new_rows)
sim_write(merged, label="MERGE")

print("After MERGE (version 3):")
sim_read().orderBy("order_id").show()

print("Updated history:")
sim_history()


In [ ]:
# Conditional MERGE: only update if status actually changed
# Delta:
#   dt.alias("t").merge(incoming2.alias("s"), "s.order_id = t.order_id")
#     .whenMatchedUpdate(condition="s.status != t.status", set={"status": "s.status"})
#     .execute()

incoming2 = spark.createDataFrame([
    ("O002", "C002", 89.5, "returned"),
    ("O006", "C004", 175.0, "delivered"),
], ["order_id", "customer_id", "amount", "status"])

current = sim_read()

joined = current.alias("t").join(incoming2.alias("s"), on="order_id", how="left")
after_cond = joined.select(
    F.col("t.order_id"),
    F.col("t.customer_id"),
    F.col("t.amount"),
    F.when(
        F.col("s.order_id").isNotNull() & (F.col("s.status") != F.col("t.status")),
        F.col("s.status")
    ).otherwise(F.col("t.status")).alias("status")
)

sim_write(after_cond, label="MERGE(conditional)")

print("After conditional MERGE (only update if status changed):")
sim_read().orderBy("order_id").show()


## Part 4: Schema Evolution

In [ ]:
# Schema enforcement: Delta rejects writes with extra columns by default
# Delta: wrong_schema.write.format("delta").mode("append").save(BASE_PATH)
#        → AnalysisException: A schema mismatch detected when writing to the Delta table

print("Schema enforcement demo:")
current = sim_read()
target_cols = set(current.columns)

wrong_schema = spark.createDataFrame([
    ("O008", "C001", 99.0, "pending", "extra_value"),
], ["order_id", "customer_id", "amount", "status", "new_col"])

extra = set(wrong_schema.columns) - target_cols
if extra:
    print(f"Schema enforcement blocked write: AnalysisException")
    print(f"(extra columns not in target schema: {extra} → rejected)")

# Schema evolution: mergeSchema=True adds the new column
# Delta: wrong_schema.write.format("delta").mode("append").option("mergeSchema","true").save(BASE_PATH)
print("\nSchema evolution with mergeSchema=True:")
current_evolved = current.withColumn("new_col", F.lit(None).cast("string"))
evolved = current_evolved.union(wrong_schema)
sim_write(evolved, label="SCHEMA_EVOLVE")

result = sim_read()
print("Schema after evolution:")
result.printSchema()
print("(existing rows have null for new_col):")
result.orderBy("order_id").show()


## Part 5: OPTIMIZE and VACUUM

In [ ]:
print("""
OPTIMIZE — Compact Small Files:
════════════════════════════════════════════════════════════════
Problem: many small Parquet files = too many file open() calls,
         slow listing, poor read performance.

# SQL
spark.sql("OPTIMIZE delta.`/path/to/table`")

# Python API
dt.optimize().executeCompaction()

# With Z-ORDER (co-locate data for common filter columns)
dt.optimize().executeZOrderBy("customer_id", "order_date")

After OPTIMIZE:
  Before: 200 × 5 MB files = 1 GB (slow)
  After:    1 × 1 GB file  = 1 GB (fast reads)

Z-ORDER: rows with the same customer_id land in the same Parquet file,
so a filter on customer_id reads fewer files.

════════════════════════════════════════════════════════════════
VACUUM — Delete Old Files:
════════════════════════════════════════════════════════════════
VACUUM removes Parquet files no longer referenced in the Delta log
AND older than the retention period.

# Default: 7 days retention
spark.sql("VACUUM delta.`/path/to/table`")

# Custom retention (minimum 168 hours enforced by default)
spark.sql("VACUUM delta.`/path/to/table` RETAIN 168 HOURS")

After VACUUM: old Parquet files deleted.
Time travel to versions older than retention → AnalysisException.

Typical schedule: run OPTIMIZE daily, VACUUM weekly.
════════════════════════════════════════════════════════════════
""")

# Simulate OPTIMIZE: compact current snapshot to 1 file
opt_path = BASE_PATH + "/optimized"
if os.path.exists(opt_path): shutil.rmtree(opt_path)
sim_read().repartition(1).write.mode("overwrite").parquet(opt_path)
pfiles = [f for f in os.listdir(opt_path) if f.endswith(".parquet")]
print(f"Simulated OPTIMIZE: compacted to {len(pfiles)} Parquet file(s)")


## Part 6: Change Data Feed

In [ ]:
# Change Data Feed (CDF): track row-level changes INSERT/UPDATE/DELETE
# Delta:
#   initial.write.format("delta").option("delta.enableChangeDataFeed","true").save(CDF_PATH)
#   spark.read.format("delta").option("readChangeFeed","true").option("startingVersion",1).load(CDF_PATH)

CDF_BASE = "/tmp/sim_cdf"
if os.path.exists(CDF_BASE): shutil.rmtree(CDF_BASE)
os.makedirs(CDF_BASE)

initial = spark.createDataFrame([
    ("O001", "pending"), ("O002", "pending"),
], ["order_id", "status"])
initial.write.mode("overwrite").parquet(CDF_BASE + "/v0")

updates = spark.createDataFrame([("O001", "delivered")], ["order_id", "status"])

before = spark.read.parquet(CDF_BASE + "/v0")
after_rows = before.join(
    updates.withColumnRenamed("status", "new_status"), on="order_id", how="left"
).withColumn(
    "status", F.coalesce(F.col("new_status"), F.col("status"))
).drop("new_status")
after_rows.write.mode("overwrite").parquet(CDF_BASE + "/v1")

# Simulate CDF: compare before vs after to find what changed
changed_ids = before.alias("b").join(
    after_rows.alias("a"), on="order_id"
).filter(F.col("b.status") != F.col("a.status")).select("order_id")

pre  = before.join(changed_ids, on="order_id", how="inner") \
             .withColumn("_change_type", F.lit("update_preimage")) \
             .withColumn("_commit_version", F.lit(1))
post = after_rows.join(changed_ids, on="order_id", how="inner") \
             .withColumn("_change_type", F.lit("update_postimage")) \
             .withColumn("_commit_version", F.lit(1))

print("Change Data Feed — what changed since version 1 (simulated):")
pre.union(post).show()
print("_change_type: 'update_preimage' = before, 'update_postimage' = after")


## Exercises

1. Create a Delta table from the orders CSV. Append 3 new orders. Then overwrite with different data. Use `.history()` to confirm 3 versions exist. Read version 0 and version 2 and compare.
2. Write a MERGE that: updates the `status` of existing orders, inserts new orders, and DELETEs orders with `status='cancelled'`.
3. Add a new column `region STRING` to an existing Delta table using `mergeSchema=True`. Verify old rows have `null` for `region`.
4. What happens if you `VACUUM` with retention 0 hours? Why does Delta block this by default?
5. Explain the difference between Delta's `overwrite` mode and a `MERGE`. When would each produce different results on a table with existing data?

In [ ]:
# Exercise 2: MERGE with delete
# Delta: .whenNotMatchedBySourceDelete(condition="t.status = 'cancelled'")

DEMO_BASE = "/tmp/sim_merge_delete"
if os.path.exists(DEMO_BASE): shutil.rmtree(DEMO_BASE)
os.makedirs(DEMO_BASE)

base = spark.createDataFrame([
    ("O001", "pending"), ("O002", "cancelled"),
    ("O003", "pending"), ("O004", "cancelled"),
], ["order_id", "status"])
base.write.mode("overwrite").parquet(DEMO_BASE + "/v0")

incoming = spark.createDataFrame([
    ("O001", "delivered"),
    ("O005", "pending"),
], ["order_id", "status"])

current = spark.read.parquet(DEMO_BASE + "/v0")

# Update matched
updated = incoming.join(current.select("order_id"), on="order_id", how="inner")

# Keep unmatched rows that are NOT cancelled (simulate whenNotMatchedBySourceDelete for cancelled)
kept = current.join(incoming.select("order_id"), on="order_id", how="left_anti") \
              .filter(F.col("status") != "cancelled")

# Insert new rows not in current
new_rows = incoming.join(current.select("order_id"), on="order_id", how="left_anti")

result = kept.union(updated).union(new_rows)
result.write.mode("overwrite").parquet(DEMO_BASE + "/v1")

print("After MERGE with delete (cancelled orders removed, simulated):")
spark.read.parquet(DEMO_BASE + "/v1").orderBy("order_id").show()
